# Módulo 05 — Criando a Primeira Rede Neural Artificial: O `Perceptron`

## 1. De Onde Surgiu o Neurônio Artificial?

Proposto em **1958** por Frank Rosenblatt, o **Perceptron** é o bloco construtivo fundamental das Redes Neurais Artificiais. Ele modela matematicamente como um neurônio biológico recebe múltiplos estímulos, pondera sua relevância e dispara (ou não) um pulso elétrico.

<div align="center">
    <img src="imagens/topologia_perceptron_simples.svg" alt="Arquitetura do Perceptron Simples" width="580">
</div>

### Analogia Direta para a Engenharia (Sistema Hidráulico + Sensor)
Observe os três componentes da figura acima como uma bancada industrial:
1. **Entradas ($x_1, x_2, \dots, x_n$):** São as **grandezas medidas pelos sensores** (pressão, acidez, temperatura).
2. **Pesos Sinápticos ($w_1, w_2, \dots, w_n$):** Funcionam como o **grau de abertura de válvulas reguladoras** em cada tubulação. Um peso alto abre totalmente a válvula (aquela variável pesa muito na decisão); um peso negativo inverte o efeito (atua como inibidor).
3. **Limiar de Ativação / Viés ($\theta = w_0$ com entrada fixa $x_0 = -1$):** É o **parafuso de calibração (*offset*) de uma válvula de alívio** — define quanta soma acumulada é necessária para o neurônio disparar.

### As Duas Equações do Perceptron
1. **Potencial de Ativação ($u$):** Soma ponderada das entradas descontando o limiar $\theta$:
$$u = \sum_{i=1}^{n} w_i x_i - \theta = w_0(-1) + w_1 x_1 + \dots + w_n x_n = \mathbf{w} \cdot \mathbf{x}$$
2. **Função de Ativação ($g(u)$):** Aplica um corte brusco (**função degrau** `{0, 1}` ou **degrau bipolar** `{-1, +1}`):
$$y = g(u) = \begin{cases} +1, & \text{se } u \ge 0 \\ -1, & \text{se } u < 0 \end{cases}$$

Vamos traduzir exatamente essa função de previsão para uma célula `Python` executável:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

def prever_perceptron(w, x):
    # 1. Calcula o potencial de ativação u = produto escalar entre pesos (w) e entradas (x)
    # Lembrando: x já inclui x0 = -1 na primeira posição para embutir o limiar w0!
    u = np.dot(w, x)

    # 2. Aplica a função degrau bipolar: retorna +1 se u >= 0, caso contrário -1
    if u >= 0.0:
        return 1
    else:
        return -1

# Teste rápido manual: entradas x1=2.0, x2=3.0 (com x0=-1 na frente)
x_exemplo = np.array([-1.0, 2.0, 3.0])
pesos_exemplo = np.array([1.5, 0.5, 0.4])  # w0=1.5 (limiar), w1=0.5, w2=0.4

print("Potencial u =", np.dot(pesos_exemplo, x_exemplo))
print("Classe prevista y =", prever_perceptron(pesos_exemplo, x_exemplo))

**Análise do cálculo:** $u = (1{,}5)(-1) + (0{,}5)(2{,}0) + (0{,}4)(3{,}0) = -1{,}5 + 1{,}0 + 1{,}2 = +0{,}7 \ge 0 \Rightarrow y = +1$.

---

## 2. Fronteira de Decisão e Separabilidade Linear

O que acontece exatamente nos pontos onde $u = 0$?
Para duas entradas ($x_1$ e $x_2$), a condição $u = 0$ fornece:
$$w_1 x_1 + w_2 x_2 - w_0 = 0 \quad \Longrightarrow \quad x_2 = -\frac{w_1}{w_2} x_1 + \frac{w_0}{w_2}$$
Isso é a **equação de uma reta** no plano $(x_1, x_2)$! (Em 3 entradas é um **plano**; em $n$ entradas é um **hiperplano**).

<div align="center">
    <img src="imagens/decision_boundary_illustration.png" alt="Fronteira de Decisão Linear" width="440">
</div>

Por isso, o Perceptron simples é um **classificador linear**: ele resolve perfeitamente qualquer problema em que exista uma linha/plano capaz de separar a Classe A (`-1`) da Classe B (`+1`).

---

## 3. Como o Neurônio Aprende Sozinho? (Regra de Aprendizado)

No início, os pesos $\mathbf{w}$ são chutados aleatoriamente (as válvulas estão desreguladas). Apresentamos todas as amostras de treino uma por uma (1 ciclo completo = **1 Época / *Epoch***).
- **Se o neurônio acertou ($y = d$):** Não mexemos em nada!
- **Se o neurônio errou ($y \neq d$):** Ajustamos as válvulas proporcionalmente ao erro $(d - y)$ e às entradas $\mathbf{x}$, modulados por uma **Taxa de Aprendizado ($\eta$, `learning_rate` entre `0` e `1`)**:
$$\mathbf{w}_{\text{novo}} = \mathbf{w}_{\text{atual}} + \eta \cdot (d - y) \cdot \mathbf{x}$$

> **Nota Conceitual Importante (Hebb vs. Rosenblatt):** No livro-base de Engenharia (*Ivan Nunes da Silva et al.*), esse ajuste supervisionado é frequentemente referido como **regra de aprendizado de Hebb** por inspirar-se no postulado biológico de Donald Hebb (1949). Em textos modernos de *Machine Learning*, você também verá o nome **Regra de Aprendizado do Perceptron (Rosenblatt)** ou **Regra Delta**, pois utiliza o erro supervisionado $(d - y)$.

<div align="center">
    <img src="imagens/decision_boundary_epochs_illustration.png" alt="Evolução da Reta por Épocas" width="440">
</div>

Vamos implementar a função completa `treinar_perceptron` e executá-la ao vivo para ensinar ao computador a porta lógica digital **`AND`** (`E` lógico)!

In [ ]:
def treinar_perceptron(X_amostras, d_desejado, taxa_aprendizado=0.1, max_epocas=100, seed=42):
    # Inicializa gerador aleatório fixo para reprodutibilidade didática
    rng = np.random.default_rng(seed)

    # Número de atributos de entrada (ex.: 2 para portas lógicas x1, x2)
    n_entradas = X_amostras.shape[1]

    # Sorteia pesos iniciais [w0, w1, ..., wn] entre -0.5 e +0.5
    w = rng.uniform(-0.5, 0.5, size=n_entradas + 1)
    historico_erros = []

    # Loop externo: Épocas (passagens completas pela tabela de treinamento)
    for epoca in range(1, max_epocas + 1):
        erros_na_epoca = 0

        # Loop interno: Avalia amostra por amostra
        for i in range(len(X_amostras)):
            # Concatena x0 = -1 na frente do vetor de entradas da amostra atual
            x_vetor = np.insert(X_amostras[i], 0, -1.0)
            d = d_desejado[i]

            # Faz a previsão com os pesos atuais
            y = prever_perceptron(w, x_vetor)

            # Regra "corrigir apenas quando errar":
            if y != d:
                w = w + taxa_aprendizado * (d - y) * x_vetor
                erros_na_epoca += 1

        historico_erros.append(erros_na_epoca)

        # Critério de convergência: zero erros na época inteira!
        if erros_na_epoca == 0:
            print(f"Convergência atingida na época {epoca}! Pesos finais: {np.round(w, 4)}")
            return w, historico_erros, True

    print(f"Atenção: Não convergiu após {max_epocas} épocas (última época teve {erros_na_epoca} erros).")
    return w, historico_erros, False

# Testando na Porta Lógica AND (bipolar: -1 = Falso, +1 = Verdadeiro)
# Só retorna +1 quando AMBAS as entradas são +1
X_logico = np.array([
    [-1.0, -1.0],
    [-1.0,  1.0],
    [ 1.0, -1.0],
    [ 1.0,  1.0]
])
d_and = np.array([-1, -1, -1, 1])

print("--- Treinando Porta AND ---")
w_and, erros_and, ok_and = treinar_perceptron(X_logico, d_and, taxa_aprendizado=0.2)

# Verificando todas as previsões após o treino
for entrada, alvo in zip(X_logico, d_and):
    pred = prever_perceptron(w_and, np.insert(entrada, 0, -1.0))
    print(f"Entrada {entrada} -> Desejado: {alvo:2d} | Previsto pelo Perceptron: {pred:2d}")

**Sucesso!** Em poucas épocas o Perceptron encontrou sozinho uma reta que acerta 100% da porta `AND`!

---

## 4. O Limite Histórico do Perceptron: O Problema do `XOR` (OU-Exclusivo)

Na porta **`XOR`** (como um interruptor paralelo *three-way* de corredor), a lâmpada acende (`+1`) quando as entradas são **diferentes** (`[-1, +1]` ou `[+1, -1]`) e apaga (`-1`) quando são **iguais** (`[-1, -1]` ou `[+1, +1]`).

Tente imaginar no plano cartesiano: é geometricamente **impossível** separar os cantos opostos de um quadrado com uma única linha reta! Veja o que acontece quando tentamos treinar o Perceptron simples no `XOR`:

In [ ]:
# Saídas desejadas da porta XOR (OU-Exclusivo)
d_xor = np.array([-1, 1, 1, -1])

print("--- Tentando treinar Porta XOR no Perceptron Simples ---")
w_xor, erros_xor, ok_xor = treinar_perceptron(X_logico, d_xor, taxa_aprendizado=0.2, max_epocas=50)

**Lição Fundamental:** O Perceptron simples **jamais converge** no `XOR` porque sua fronteira é estritamente linear. É exatamente para quebrar essa barreira que estudaremos o **Perceptron Multicamadas (PMC)** no **Módulo 06**!

---

## 5. Efeito da Taxa de Aprendizado ($\eta$) na Estabilidade

A taxa de aprendizado $\eta$ funciona como o ganho proporcional $K_p$ de um controlador industrial:
- **$\eta$ excessivamente alta:** Grandes saltos nos pesos fazem a fronteira oscilar bruscamente de uma época para outra.
- **$\eta$ adequada:** Ajustes suaves guiam a reta até separar limpa e estavelmente as duas classes.

Veja a evolução real das fronteiras de decisão treinadas pelo próprio algoritmo sobre duas nuvens de dados:

In [ ]:
# Gerando duas classes sintéticas linearmente separáveis em 2D
rng_vis = np.random.default_rng(42)
classe_a = rng_vis.normal(loc=2.0, scale=0.7, size=(25, 2))  # Classe -1 centrada em (2, 2)
classe_b = rng_vis.normal(loc=5.5, scale=0.7, size=(25, 2))  # Classe +1 centrada em (5.5, 5.5)

X_sint = np.vstack([classe_a, classe_b])
d_sint = np.hstack([-np.ones(25, dtype=int), np.ones(25, dtype=int)])

def historico_retas_treino(X, d, lr, n_epocas=6):
    # Registra os pesos ao fim de cada época para visualizar o movimento real da reta
    w = np.array([6.0, -0.8, 1.0])  # Ponto de partida ruim (reta mal posicionada)
    retas = [w.copy()]
    for _ in range(n_epocas):
        for i in range(len(X)):
            x_v = np.insert(X[i], 0, -1.0)
            y = prever_perceptron(w, x_v)
            if y != d[i]:
                w = w + lr * (d[i] - y) * x_v
        retas.append(w.copy())
    return retas

retas_alta = historico_retas_treino(X_sint, d_sint, lr=2.5, n_epocas=4)
retas_boa = historico_retas_treino(X_sint, d_sint, lr=0.05, n_epocas=4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
eixo_x = np.linspace(0, 8, 100)

for ax, lista_w, titulo in [
    (ax1, retas_alta, "Taxa Alta (lr = 2.5): Saltos Bruscos"),
    (ax2, retas_boa, "Taxa Adequada (lr = 0.05): Convergência Suave")
]:
    ax.scatter(classe_a[:, 0], classe_a[:, 1], color="royalblue", label="Classe A (-1)")
    ax.scatter(classe_b[:, 0], classe_b[:, 1], color="darkorange", marker="^", label="Classe B (+1)")
    estilos = ["r--", "g-.", "b:", "m-", "k-"]
    for ep_idx, w_ep in enumerate(lista_w):
        if abs(w_ep[2]) > 1e-6:
            # Da equação w1*x1 + w2*x2 - w0 = 0  =>  x2 = (w0 - w1*x1) / w2
            eixo_y = (w_ep[0] - w_ep[1] * eixo_x) / w_ep[2]
            rotulo = "Inicial" if ep_idx == 0 else f"Época {ep_idx}"
            ax.plot(eixo_x, eixo_y, estilos[ep_idx % len(estilos)], label=rotulo, alpha=0.85)
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 8)
    ax.set_title(titulo)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

---

## 6. Perguntas Frequentes de Prova e Projeto (FAQ)

1. **Por que fixamos $x_0 = -1$ e tratamos o limiar $\theta$ como mais um peso $w_0$?**
   Para transformar $w_1 x_1 + w_2 x_2 - \theta$ em um **único produto escalar vetorial** $\mathbf{w} \cdot \mathbf{x}$ e atualizar $\theta$ automaticamente com a mesmíssima linha de código `w += lr * (d - y) * x`!
2. **Duas redes Perceptron treinadas para o mesmo problema chegam sempre na mesma reta?**
   Não! Se duas nuvens de pontos têm um "corredor vazio" entre elas, existem **infinitas retas** que passam por esse corredor sem errar nenhum ponto. A reta escolhida depende do sorteio inicial dos pesos e da taxa $\eta$.
3. **Como saber na prática se um conjunto de dados real (em 10 dimensões, impossível de desenhar) é linearmente separável?**
   Rode o Perceptron com um limite alto de épocas: se o número de erros cair até **exatamente zero**, os dados são linearmente separáveis; se o erro continuar oscilando indefinidamente, você precisa de um **PMC (MLP)**.

---

## 7. Exercícios Práticos de Fixação

### Exercício 1: Treinando a Porta Lógica `OR` (OU Inclusivo)
Na porta `OR` bipolar, a saída só é `-1` quando **ambas** as entradas são `-1`; caso contrário (`[-1, 1]`, `[1, -1]`, `[1, 1]`), a saída é `+1`. Use a função `treinar_perceptron` definida acima para treinar a porta `OR` e verifique todas as 4 saídas.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
# Tabela-verdade bipolar da porta OR: [-1, +1, +1, +1]
d_or = np.array([-1, 1, 1, 1])

w_or, erros_or, ok_or = treinar_perceptron(X_logico, d_or, taxa_aprendizado=0.2)

for entrada, alvo in zip(X_logico, d_or):
    pred = prever_perceptron(w_or, np.insert(entrada, 0, -1.0))
    print(f"Entrada {entrada} -> Desejado: {alvo:2d} | Previsto: {pred:2d}")
```

</details>

### Exercício 2: Alarme Industrial de Sobrecarga Termomecânica
Um mancal industrial entra em regime de falha (`+1`) quando a combinação de temperatura normalizada $x_1$ e vibração normalizada $x_2$ ultrapassa um limite seguro, e opera normal (`-1`) caso contrário. Dados:
- Normais (`-1`): `[0.2, 0.1]`, `[0.3, 0.2]`, `[0.1, 0.4]`
- Falha (`+1`): `[0.8, 0.7]`, `[0.9, 0.6]`, `[0.7, 0.9]`
Treine o Perceptron e preveja o status de uma nova leitura `[0.85, 0.80]`.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
X_mancal = np.array([
    [0.2, 0.1], [0.3, 0.2], [0.1, 0.4],  # Operação normal (-1)
    [0.8, 0.7], [0.9, 0.6], [0.7, 0.9]   # Regime de falha (+1)
])
d_mancal = np.array([-1, -1, -1, 1, 1, 1])

w_mancal, _, _ = treinar_perceptron(X_mancal, d_mancal, taxa_aprendizado=0.1)

nova_leitura = np.array([-1.0, 0.85, 0.80])  # Não esqueça do x0 = -1 na frente!
status = prever_perceptron(w_mancal, nova_leitura)
print("Diagnóstico da nova leitura:", "FALHA (+1)" if status == 1 else "NORMAL (-1)")
```

</details>

### Exercício 3: Equação Analítica da Reta Encontrada
Pegue o vetor `w_and = [w0, w1, w2]` obtido no treinamento da porta `AND` e imprima o coeficiente angular $a = -w_1 / w_2$ e o coeficiente linear $b = w_0 / w_2$ da reta separadora $x_2 = a x_1 + b$.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
w0, w1, w2 = w_and
coef_angular = -w1 / w2
coef_linear = w0 / w2

print(f"Equação da fronteira de decisão: x2 = ({coef_angular:.4f}) * x1 + ({coef_linear:.4f})")
```

</details>

---

## Próximo Passo

Vimos na prática o poder e a limitação geométrica de um único neurônio. Agora vamos conectar vários neurônios em camadas e vencer o problema do `XOR` no **[Módulo 06 — Perceptron Multicamadas (PMC) e Backpropagation](06_pmc_e_backpropagation.ipynb)**!